# Using Alternative LLM Providers

*Extending the book's examples with OpenAI-compatible providers*

<a href="https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961"><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="https://github.com/HandsOnLLM/Hands-On-Large-Language-Models"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>

---

Throughout the book, we use [OpenAI's API](https://openai.com/) in Chapters 4, 5, and 7 for text classification, topic modeling, and agents. One of the great things about the OpenAI SDK is that many other LLM providers offer **OpenAI-compatible APIs**. This means you can use the **exact same code** from the book with different providers by simply changing the `base_url` and `api_key`.

This notebook demonstrates how to use [MiniMax](https://www.minimaxi.com/) as an alternative provider. MiniMax offers the **MiniMax-M3** flagship model with enhanced reasoning and coding capabilities through an OpenAI-compatible endpoint. The same approach works for other OpenAI-compatible providers as well.

---

### [OPTIONAL] - Installing Packages on <img src="https://colab.google/static/images/icons/colab.png" width=100>

If you are viewing this notebook on Google Colab, you need to **uncomment and run** the following codeblock to install the dependencies:

In [ ]:
# %%capture
# !pip install openai langchain langchain_openai

## Swapping Providers with the OpenAI SDK

In **Chapter 4**, we create an OpenAI client like this:

```python
import openai
client = openai.OpenAI(api_key="YOUR_KEY_HERE")
```

To use MiniMax instead, we simply set the `base_url` parameter:

In [ ]:
import openai

# MiniMax: OpenAI-compatible endpoint
client = openai.OpenAI(
    api_key="YOUR_MINIMAX_KEY",  # Get your key at https://www.minimaxi.com/
    base_url="https://api.minimax.io/v1"
)

That's it! The rest of the code from the book stays **exactly the same**. Let's verify with the text classification example from Chapter 4:

In [ ]:
def chatgpt_generation(prompt, document, model="MiniMax-M3"):
    """Generate an output based on a prompt and an input document.

    This is the same function from Chapter 4, with only the default
    model name changed to use MiniMax.
    """
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt.replace("[DOCUMENT]", document)},
    ]
    chat_completion = client.chat.completions.create(
        messages=messages,
        model=model,
        temperature=0.01,  # MiniMax requires temperature > 0
    )
    return chat_completion.choices[0].message.content

In [ ]:
# Same prompt from Chapter 4
prompt = """Predict whether the following document is a positive or negative movie review:

[DOCUMENT]

If it is positive return 1 and if it is negative return 0. Do not give any other answers.
"""

document = "unpretentious , charming , quirky , original"
chatgpt_generation(prompt, document)

## Using MiniMax with LangChain

In **Chapter 7**, we use LangChain's `ChatOpenAI` for agents and chains. Since MiniMax provides an OpenAI-compatible API, we can use it directly with `ChatOpenAI` by specifying the `openai_api_base`:

In [ ]:
from langchain_openai import ChatOpenAI

# Use MiniMax as the LLM backend in LangChain
llm = ChatOpenAI(
    model_name="MiniMax-M3",
    openai_api_key="YOUR_MINIMAX_KEY",
    openai_api_base="https://api.minimax.io/v1",
    temperature=0.01,
)

In [ ]:
# All LangChain patterns from Chapter 7 work as-is
response = llm.invoke("What is the capital of France?")
print(response.content)

All the LangChain patterns from Chapter 7 (prompt templates, chains, memory, and agents) work without any modification. You only need to change the LLM initialization.

## Using Environment Variables

For a cleaner setup, you can configure the provider through environment variables. This makes it easy to switch between providers without changing your code:

In [ ]:
import os
import openai

# Configure via environment variables
# For OpenAI (default):
#   export LLM_API_KEY="sk-..."
#   export LLM_BASE_URL="https://api.openai.com/v1"
#   export LLM_MODEL="gpt-3.5-turbo"
#
# For MiniMax:
#   export LLM_API_KEY="your-minimax-key"
#   export LLM_BASE_URL="https://api.minimax.io/v1"
#   export LLM_MODEL="MiniMax-M3"

client = openai.OpenAI(
    api_key=os.environ.get("LLM_API_KEY", "YOUR_KEY_HERE"),
    base_url=os.environ.get("LLM_BASE_URL", "https://api.openai.com/v1"),
)
model = os.environ.get("LLM_MODEL", "gpt-3.5-turbo")

## Summary

Because many LLM providers now offer OpenAI-compatible APIs, you can extend the book's examples to use different models with minimal code changes:

| Provider | `base_url` | Example Model |
|----------|-----------|---------------|
| [OpenAI](https://openai.com/) | `https://api.openai.com/v1` (default) | `gpt-3.5-turbo`, `gpt-4o` |
| [MiniMax](https://www.minimaxi.com/) | `https://api.minimax.io/v1` | `MiniMax-M3`, `MiniMax-M2.7`, `MiniMax-M2.7-highspeed` |

The key takeaway: **focus on learning the patterns and concepts from the book** — they transfer across providers. Once you understand prompt engineering, chains, memory, and agents, you can use them with any compatible LLM.